In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
from load_config import load_config
from sklearn.metrics.pairwise import cosine_similarity
import json

config = load_config()

file_path1 = config['data']['file1_path']
file_path2 = config['data']['file2_path']

df_mv = pd.read_csv(file_path1)
df_cr = pd.read_csv(file_path2)

df_new=df_cr.merge(df_mv,on='title')

df = df_new[['genres','id','keywords','title','overview','production_companies','cast','crew']]

df['genres'] =  df['genres'].apply(lambda x :[i['name'] for i in json.loads(x)])
df['keywords'] =  df['keywords'].apply(lambda x :[i['name'] for i in json.loads(x)])
df['production_companies'] =  df['production_companies'].apply(lambda x :[i['name'] for i in json.loads(x)])
df['cast'] =df['cast'].apply(lambda x:[i['name'] for i in json.loads(x)[:5]])
df['crew'] =df['crew'].apply(lambda x:[i['name'] for i in json.loads(x) if i['job']=='Director'])


df['overview'] = df['overview'].fillna('')
df['overview'] = df['overview'].str.split()


df['genres'] =  df['genres'].apply(lambda x :[ i.replace(" ",'') for i in x])
df['production_companies'] =  df['production_companies'].apply(lambda x :[ i.replace(" ",'') for i in x])
df['cast'] =df['cast'].apply(lambda x :[ i.replace(" ",'') for i in x])
df['crew'] =df['crew'].apply(lambda x :[ i.replace(" ",'') for i in x])
df['keywords'] =df['keywords'].apply(lambda x :[ i.replace(" ",'') for i in x])

df['overview'] =df['overview'].apply(lambda x :[ i.replace(" ",'') for i in x])


df['tags'] = df['overview'] + df['keywords'] +df['genres'] +df['production_companies'] + df['cast'] + df['crew']
df['tags'] = df['tags'].apply(lambda x: ' '.join(x))
df.drop(['overview','keywords','genres','production_companies','cast','crew'], axis =1 , inplace = True)

ct_vector = CountVectorizer(stop_words='english',max_features=50000)
count_vector= ct_vector.fit_transform(df['tags']).toarray()

tdidf_vector = TfidfVectorizer(stop_words='english',max_features=50000)
tfidf= tdidf_vector.fit_transform(df['tags']).toarray()

similar_ct_vector = cosine_similarity(count_vector)
similar_tdidf_vector = cosine_similarity(tfidf)

def recommend_count_vector(movie):

				movie_index = df[df['title'].str.lower() == movie.lower()].index[0]
				distance = similar_ct_vector[movie_index]

				movie_list= sorted(list(enumerate(distance)), reverse=True,key = lambda x:x[1])[1:6]

				for i in movie_list:
					print(f'Count vector',df.iloc[i[0]].title, round(i[1],3))

recommend_count_vector('Avatar')

def recommend_TFDIF(movie):

				movie_index = df[df['title'].str.lower() == movie.lower()].index[0]
				distance = similar_tdidf_vector[movie_index]

				movie_list= sorted(list(enumerate(distance)), reverse=True,key = lambda x:x[1])[1:6]

				for i in movie_list:
					print(f'using TDIDF',df.iloc[i[0]].title, round(i[1],3))

recommend_TFDIF('Avatar')

